# Part A · Caltech-256 training: ResNet-10t and ResNet-200d (Kaggle)

This notebook only runs `train/hp_tune.py` (LP → FT) for our two sidekicks, in one run. Logits, temperature scaling and evaluation run from the `partA-kaggle` notebook once everyone's checkpoints are in.

**Notebook settings:** Accelerator *GPU T4 ×2*, Internet *on* (needs a phone-verified account). Add the secret `GH_TOKEN` under *Add-ons → Secrets*.

**Run it with Save Version → Save & Run All.** Kaggle keeps `/kaggle/working` (`checkpoints/`, `configs/`) only for a saved version, and a version may run for at most 12 h. To continue from an earlier version, add its output as an input and set `PREVIOUS_OUTPUT`: models whose LP checkpoint exists skip LP.

In [ ]:
from pathlib import Path
import torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
OUT = Path('/kaggle/working')
PREVIOUS_OUTPUT = None              # e.g. Path('/kaggle/input/<this-notebook>') to resume from an earlier version
N_GPUS = torch.cuda.device_count()  # 2 on T4 x2

REPO_URL = "https://github.com/gorrast/DD2610-Final-Project.git"
GITHUB_TOKEN_SECRET = "GH_TOKEN"

## 0 · Code and environment

In [ ]:
import os, sys, shutil
from base64 import b64encode

# The repo lives in /tmp so Ray's trial checkpoints (ray_results/) are not saved as notebook output.
auth = b64encode(f"x-access-token:{secrets.get_secret(GITHUB_TOKEN_SECRET)}".encode()).decode()
!rm -rf /tmp/repo && git -c http.extraHeader="Authorization: Basic {auth}" clone -q --depth 1 -b resnet10t-200d {REPO_URL} /tmp/repo
%cd /tmp/repo/original_code
if PREVIOUS_OUTPUT:
    shutil.copytree(PREVIOUS_OUTPUT, OUT, dirs_exist_ok=True)
for d in ['checkpoints', 'y-prediction', 'evaluation', 'configs']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
    os.symlink(OUT / d, d)

# Scripts in train/ import the package folders, so they need original_code on the path.
sys.path.insert(0, os.getcwd())
os.environ.update(PYTHONPATH=os.getcwd(), WANDB_MODE='disabled', PYTHONUNBUFFERED='1')
!pip install -q wilds timm "ray[tune]" fvcore

## 1 · Data
Only Caltech-256 is needed for training. It goes to `/tmp/data` on local disk, outside the saved output. The split check uses the authors' seed-42 loader.

In [ ]:
from torchvision.datasets import Caltech256
from load_data import get_dataloaders
from load_models.transforms import get_transforms

DATA = Path('/tmp/data')
Caltech256(DATA, download=True)
caltech = get_dataloaders('caltech256', DATA, 64, 2, get_transforms())
print({k: len(v.dataset) for k, v in caltech.items() if v})   # train 21424, val 4591, test 4592

## 2 · Training config
`caltech256.yaml` is missing from the release. Batch size 128, 8 LP epochs and 16 FT epochs come from paper Appendix A.2. `warmup_epochs` and `grace_period` are **our choice**, because the paper doesn't give them. `num_workers` is split between the GPUs, because two trials share Kaggle's 4 CPUs.

In [ ]:
import yaml

yaml.safe_dump({'training': {'batch_size': 128, 'num_epochs_lp': 8, 'num_epochs_ff': 16,
                             'num_workers': os.cpu_count() // N_GPUS, 'warmup_epochs': 2, 'grace_period': 4}},
               open('configs/caltech256.yaml', 'w'))

# (model_name, source, tv_weights); the smaller model first, so it finishes even if ResNet-200d runs long
MODELS = [('resnet10t.c3_in1k', 'timm', None),
          ('resnet200d.ra2_in1k', 'timm', None)]

## 3 · LP → FT  →  `train/hp_tune.py`
**Our reduced protocol** (fixed before looking at any test results): a grid search over the learning rate only, with 4 points per stage. Each point runs once. We chose the points from the validation results of the ResNet-18 sweep (`local_3080.ipynb`), around its best LP and FT learning rates and below FT's drop-off between 1.2e-4 and 1.85e-4. All 4 points stay inside the authors' search ranges.
* **LP:** learning rate ∈ {1.4e-3, 1.8e-3, 2.5e-3, 3.4e-3}, weight decay 0.
* **FT:** learning rate ∈ {3.7e-5, 5.9e-5, 9.4e-5, 1.5e-4}, weight decay **fixed at 1e-6** (authors searched [1e-8, 1e-5]).

`hp_tune.py` is unchanged from the authors' release. It builds its search space with `tune.loguniform` and starts it with `tune.run`, both looked up on Ray's `tune` module when called, so the cell below swaps those two functions for our grid before running the script. We start Ray with 4 logical CPUs per GPU, so on T4 ×2 two trials run at once. ASHA can still stop weak FT trials early.

**Time budget:** our estimate is ~1.2 h for ResNet-10t and ~12 h for ResNet-200d, right at Kaggle's 12 h limit. Watch ResNet-200d's first FT epoch: if it takes more than about 20 min, the run won't finish.

In [ ]:
import ray, runpy, sys, time
from ray import tune
from ray.tune.tune import run as tune_run   # Ray's own run, so rerunning this cell stays safe

LP_LRS = [1.4e-3, 1.8e-3, 2.5e-3, 3.4e-3]   # chosen from the ResNet-18 LP sweep
FT_LRS = [3.7e-5, 5.9e-5, 9.4e-5, 1.5e-4]   # chosen from the ResNet-18 FT sweep
FT_WD = 1e-6                            # fixed; the authors searched [1e-8, 1e-5]
SPACE = {(1e-4, 1e-2): tune.grid_search(LP_LRS),   # keyed by the ranges written in hp_tune.py
         (1e-6, 3e-4): tune.grid_search(FT_LRS),
         (1e-8, 1e-5): FT_WD}
tune.loguniform = lambda low, high: SPACE[(low, high)]
tune.run = lambda *args, **kwargs: tune_run(*args, **{**kwargs, 'num_samples': 1})   # each grid point once

!ray stop > /dev/null 2>&1
ray.init(address='local', num_cpus=4 * N_GPUS, num_gpus=N_GPUS, include_dashboard=False)
for name, source, weights in MODELS:
    t0 = time.time()
    sys.argv = ['hp_tune.py', '--model_name', name, '--source', source, '--tv_weights', str(weights),
                '--dataset_name', 'caltech256', '--dataset_dir', str(DATA)]
    runpy.run_path('train/hp_tune.py', run_name='__main__')   # runs in this kernel, so the swap applies
    print(f'{name}: {(time.time() - t0) / 3600:.1f} h')